# ansel-denoise — training on Kaggle

Trains the CFA-domain denoising U-Net on the shard cache published on the
[`shards-v1` release](https://github.com/aurelienpierreeng/ansel-denoise/releases/tag/shards-v1),
with noise synthesized on the fly from the darktable/Ansel noise profiles.
Kaggle twin of `colab_train.ipynb`: same one-shot fixed cosine schedule,
same resume-until-done contract, Kaggle-native persistence.

**Before running (Settings panel, right side):**
- *Accelerator* → **GPU T4 ×2** (or P100). Kaggle gives ~30 GPU-hours/week.
- *Internet* → **On** (needed to clone the repo and fetch the shards).
- Optional but recommended — checkpoint persistence across sessions:
  *Add-ons → Secrets*, add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from
  kaggle.com → Settings → API → Create New Token). The notebook then pushes
  checkpoints to a private Kaggle Dataset after each session, and any later
  session resumes from it once you attach that dataset via *Add Input*.

**Two ways to run:**
- *Interactive*: run all cells; on disconnect, run all cells again — training
  resumes from the latest checkpoint (session-local, attached dataset, or
  pushed dataset).
- *Commit* (“Save & Run All”): runs headless up to the session cap;
  `TIME_BUDGET_H` stops training early enough for the checkpoint-push and
  output-save to happen. Everything in `/kaggle/working` is also kept as the
  version output — a second, automatic persistence layer.

To **start over cold** (new dataset snapshot, changed hyperparameters), pick
a new `RUN_NAME`.

In [ ]:
# --- parameters -----------------------------------------------------------
TOTAL_STEPS = 100_000    # full run length; sessions resume until this is reached
BATCH = 32
PATCH = 128
RUN_NAME = "v1-100k"     # change to start a fresh run (cold restart)
CKPT_DATASET = f"ansel-denoise-{RUN_NAME}"  # per-run private checkpoint dataset
TIME_BUDGET_H = 8.0      # stop training this many hours in, leaving time to push/save

!nvidia-smi -L || echo 'NO GPU: set Settings -> Accelerator -> GPU'

In [ ]:
# --- code -----------------------------------------------------------------
# Same contract as the Colab notebook: absolute path, pulled (not skipped)
# when already cloned so a reused kernel runs the CURRENT code; already-
# imported modules purged before re-import. Dependencies (numpy, torch) are
# preinstalled on Kaggle images; src/ on sys.path is enough.
import os, sys
REPO = '/kaggle/tmp/ansel-denoise'   # ephemeral: keeps the committed output lean
os.makedirs('/kaggle/tmp', exist_ok=True)
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 https://github.com/aurelienpierreeng/ansel-denoise.git {REPO}
!git -C {REPO} log --oneline -1
%cd {REPO}
sys.path.insert(0, os.path.join(REPO, 'src'))
for m in [m for m in list(sys.modules) if m.startswith('ansel_denoise')]:
    del sys.modules[m]
import ansel_denoise
print('package importable:', ansel_denoise.__file__)

In [ ]:
# --- data: attached dataset if present, else fetch the public release -----
# Fast path: publish the shards ONCE as a Kaggle Dataset and attach it via
# Add Input — startup then skips the multi-GB download. Fallback: fetch the
# release assets into ephemeral /kaggle/tmp (no auth, public release).
import json, tarfile, urllib.request
from pathlib import Path

attached = sorted(Path('/kaggle/input').glob('*/**/*.npz')) if Path('/kaggle/input').exists() else []
if attached:
    SHARDS = attached[0].parent
    print(f'using attached shard dataset: {SHARDS} ({len(list(SHARDS.glob("*.npz")))} shards)')
else:
    SHARDS = Path('/kaggle/tmp/shards')
    SHARDS.mkdir(parents=True, exist_ok=True)
    api = 'https://api.github.com/repos/aurelienpierreeng/ansel-denoise/releases/tags/shards-v1'
    with urllib.request.urlopen(api) as r:
        assets = json.load(r)['assets']
    for a in assets:
        if not a['name'].startswith('shards-'):
            continue
        print(a['name'], f"{a['size'] / 1e6:.0f} MB")
        tmp, _ = urllib.request.urlretrieve(a['browser_download_url'])
        with tarfile.open(tmp) as tar:
            tar.extractall(SHARDS, filter='data')
        Path(tmp).unlink()
    print(f"{len(list(SHARDS.glob('*.npz')))} shards ready")

In [ ]:
# --- checkpoints: working dir + any attached checkpoint dataset -----------
# /kaggle/working persists as the committed version's output; the pushed
# checkpoint dataset (cell below) covers interactive sessions. Resume from
# the highest NUMBERED checkpoint anywhere (never ckpt-final.pt: a stable
# alias that can be staler than a mid-run numbered checkpoint).
from pathlib import Path

OUT = Path('/kaggle/working/runs') / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)

candidates = list(OUT.glob('ckpt-0*.pt'))
if Path('/kaggle/input').exists():
    # pushed checkpoint datasets are flat; version outputs keep runs/RUN_NAME/
    candidates += list(Path('/kaggle/input').glob('*/ckpt-0*.pt'))
    candidates += list(Path('/kaggle/input').glob(f'*/runs/{RUN_NAME}/ckpt-0*.pt'))
candidates.sort(key=lambda p: int(p.stem.split('-')[1]))
start = int(candidates[-1].stem.split('-')[1]) if candidates else 0
RESUME = ['--resume', str(candidates[-1])] if candidates else []
print(f'checkpoints -> {OUT} | resume from step {start} of {TOTAL_STEPS}'
      + (' — run complete, cells below just re-export' if start >= TOTAL_STEPS else ''))

In [ ]:
# --- train, under a wall-clock budget -------------------------------------
# Kaggle kills sessions hard at the cap; training runs as a subprocess with a
# timeout so the push/export cells below ALWAYS get their turn. A timeout is
# not a failure: the last numbered checkpoint (written every 500 steps) is
# the resume anchor for the next session — same fixed cosine schedule.
import subprocess, sys, os

env = dict(os.environ, PYTHONPATH=os.path.join(REPO, 'src'))
cmd = [sys.executable, '-m', 'ansel_denoise.train',
       '--shards', str(SHARDS), '--out', str(OUT),
       '--steps', str(TOTAL_STEPS), '--batch', str(BATCH), '--patch', str(PATCH),
       '--workers', '2', '--val-every', '500', '--ckpt-every', '500',
       '--schedule', 'cosine', *RESUME]
print(' '.join(cmd))
try:
    subprocess.run(cmd, env=env, cwd=REPO, timeout=TIME_BUDGET_H * 3600, check=True)
    print('training reached TOTAL_STEPS')
except subprocess.TimeoutExpired:
    print(f'time budget ({TIME_BUDGET_H} h) reached — checkpoints are saved, '
          f'rerun (or re-commit) this notebook to resume')

In [ ]:
# --- persist checkpoints to a private Kaggle Dataset ----------------------
# Uses the KAGGLE_USERNAME / KAGGLE_KEY secrets (Add-ons -> Secrets). Skipped
# gracefully without them — the committed version output still carries
# /kaggle/working. Only the newest numbered checkpoint, ckpt-best, ckpt-final
# and the log are pushed. Files are staged FLAT: the kaggle CLI's default
# dir-mode skips subdirectories, and the per-run dataset slug keeps runs from
# mixing their resume anchors.
import json, shutil, subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    user, key = s.get_secret('KAGGLE_USERNAME'), s.get_secret('KAGGLE_KEY')
except Exception as e:
    user = None
    print(f'no Kaggle API secrets ({e}) — relying on the committed version output only')

if user:
    Path('/root/.kaggle').mkdir(exist_ok=True)
    Path('/root/.kaggle/kaggle.json').write_text(json.dumps({'username': user, 'key': key}))
    Path('/root/.kaggle/kaggle.json').chmod(0o600)

    stage = Path('/kaggle/tmp/ckpt-push')
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True)
    numbered = sorted(OUT.glob('ckpt-0*.pt'))
    for f in ([numbered[-1]] if numbered else []) +              [OUT / 'ckpt-best.pt', OUT / 'ckpt-final.pt', OUT / 'train.log']:
        if f.exists():
            shutil.copy2(f, stage / f.name)
    (stage / 'dataset-metadata.json').write_text(json.dumps({
        'title': CKPT_DATASET, 'id': f'{user}/{CKPT_DATASET}',
        'licenses': [{'name': 'CC0-1.0'}]}))
    step = numbered[-1].stem.split('-')[1] if numbered else '0'
    r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(stage),
                        '-m', f'{RUN_NAME} step {step}', '-q'], capture_output=True, text=True)
    if r.returncode != 0:  # first push: the dataset does not exist yet
        r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(stage), '-q'],
                           capture_output=True, text=True)
    print(r.stdout or r.stderr)
    print(f'-> attach kaggle.com/datasets/{user}/{CKPT_DATASET} via Add Input '
          f'so the next session can resume from it')

In [ ]:
# --- export the weights for Ansel ----------------------------------------
# Lands in /kaggle/working -> downloadable from the notebook Output tab.
from ansel_denoise.export import main as export_main
ckpt = OUT / 'ckpt-final.pt'
if ckpt.exists():
    export_main([str(ckpt), '--out', '/kaggle/working/final.anselnn'])
    print('weights: /kaggle/working/final.anselnn (Output tab)')
else:
    print('run not finished yet — no ckpt-final.pt; resume until TOTAL_STEPS')